### Notebook 02 — Data Cleaning & Feature Engineering

**Purpose:**
Transform raw OHLCV data into a rich analytical dataset.

**Features engineered:**
- Daily return and cumulative return
- Moving averages: 20-day, 50-day, 200-day
- Rolling 30-day volatility (annualized)
- RSI — Relative Strength Index (14-day)
- VWAP — Volume Weighted Average Price
- Bollinger Bands (20-day, 2 standard deviations)
- Drawdown from rolling 52-week high
- Binary signal flags: above MA50, above MA200, golden cross
- RSI zone classification
- Date dimension columns for Power BI time intelligence

In [2]:
# CELL 2 — Imports

import os
import sys
import logging
import warnings

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath(".."))
from config.settings import PROCESSED_DATA_PATH, LOG_PATH

# Logging — append to existing log
logging.basicConfig(
    level = logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(f"../{LOG_PATH}", mode="a"),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("cleaning")

# Load master dataset
master_df = pd.read_csv(
    f'../{PROCESSED_DATA_PATH}master_stock_data.csv',
    parse_dates = ['date']
)

logger.info(f"Loaded master dataset: {master_df.shape[0]:,} rows, "
            f"{master_df['ticker'].nunique()} tickers")

print(f"Shape: {master_df.shape}")
print(f"Columns: {list(master_df.columns)}")
print(master_df.head(3).to_string(index=False))

2026-05-24 19:24:54 | INFO | Loaded master dataset: 19,272 rows, 12 tickers
Shape: (19272, 9)
Columns: ['date', 'ticker', 'company', 'sector', 'open', 'high', 'low', 'close', 'volume']
      date ticker    company     sector      open      high       low     close    volume
2020-01-02   AAPL Apple Inc. Technology 71.344054 72.394086 71.091184 72.333878 135480400
2020-01-03   AAPL Apple Inc. Technology 71.563221 72.389273 71.406681 71.630653 146322800
2020-01-06   AAPL Apple Inc. Technology 70.753991 72.239919 70.503524 72.201385 118387200


In [3]:
# CELL 3 — Feature Engineering Function

def engineer_features(df):
    """
    Calculates all financial features for a single ticker's dataframe.

    Input:
        df — dataframe for ONE ticker, will be sorted inside

    Returns:
        df — same dataframe with all features added
    """
    df = df.copy()

    df = df.sort_values('date').reset_index(drop = True)

    # ── 1. Returns ────────────────────────────────────────────
    # Daily return: (today_close - yesterday_close) / yesterday_close * 100

    df['daily_return'] = df['close'].pct_change() * 100

    # Cumulative return from start of dataset
    # Formula: (current_price / first_price - 1) * 100
    first_price = df['close'].iloc[0]
    df['cumulative_return'] = ((df['close']/first_price)-1) * 100

    # ── 2. Moving Averages ────────────────────────────────────
    # Used to identify trend direction

    df['ma_20'] = df['close'].rolling(window = 20, min_periods = 1).mean().round(2)
    df["ma_50"]  = df["close"].rolling(window=50,  min_periods=1).mean().round(2)
    df["ma_200"] = df["close"].rolling(window=200, min_periods=1).mean().round(2)

    # ── 3. Volatility ─────────────────────────────────────────
    # 30-day rolling standard deviation of daily returns
    # Represents short-term risk

    df['volatility_30d'] = (
        df['daily_return'].rolling(window = 30, min_periods = 5).std()).round(4)
    
    # Annualized volatility — multiply by sqrt(252) trading days per year
    
    df['volatility_ann'] = (
        df['volatility_30d'] * np.sqrt(252)
    ).round(4)

    # ── 4. RSI — Relative Strength Index (14-day) ─────────────
    # Momentum oscillator: measures speed and magnitude of price changes
    # RSI > 70 = overbought (potential sell signal)
    # RSI < 30 = oversold  (potential buy signal)
    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=14, min_periods=1).mean()
    avg_loss = loss.rolling(window=14, min_periods=1).mean()
    rs = avg_gain/avg_loss.replace(0, np.nan)
    df['rsi_14'] = (100 -(100/ (1+rs))).round(2)
    df['rsi_14'] = df['rsi_14'].fillna(50)  # neutral RSI when undefined

    # ── 5. VWAP — Volume Weighted Average Price ───────────────
    # Shows average price weighted by volume over 20 days
    df['vwap'] = (
        (df['close'] * df['volume']).rolling(window = 20, min_periods = 1).sum()
        / df['volume'].rolling(window = 20, min_periods = 1).sum()
    ).round(2)

    # ── 6. Bollinger Bands ────────────────────────────────────
    # 20-day MA ± 2 standard deviations
    # Price outside bands signals unusual market activity

    rolling_std = df['close'].rolling(window = 20, min_periods = 1).std()
    df['bb_upper'] = (df['ma_20'] + 2 * rolling_std).round(2)
    df["bb_lower"] = (df["ma_20"] - 2 * rolling_std).round(2)
    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]).round(2)

    # Bollinger Band position (0 = at lower band, 1 = at upper band)
    bb_range = (df["bb_upper"] - df["bb_lower"]).replace(0, np.nan)
    df["bb_pct"] = ((df["close"] - df["bb_lower"]) / bb_range).round(4)

    # ── 7. Drawdown ───────────────────────────────────────────
    # How far the price has fallen from its rolling 252-day peak
    rolling_max   = df["close"].rolling(window=252, min_periods=1).max()
    df["drawdown"] = ((df["close"] - rolling_max) / rolling_max * 100).round(4)

    # ── 8. Signal Flags ───────────────────────────────────────
    # Binary columns for SQL CASE WHEN and Power BI conditional formatting

    # Price above key moving averages
    df["above_ma50"]  = (df["close"] > df["ma_50"]).astype(int)
    df["above_ma200"] = (df["close"] > df["ma_200"]).astype(int)

    # Golden Cross: MA50 crosses above MA200 — classic bullish signal
    # Death Cross: MA50 crosses below MA200 — classic bearish signal
    df["golden_cross"] = (df["ma_50"] > df["ma_200"]).astype(int)

    df["rsi_zone"] = pd.cut(
        df["rsi_14"],
        bins=[0, 30, 45, 55, 70, 100],
        labels=["Oversold", "Bearish", "Neutral", "Bullish", "Overbought"],
        right=False
    ).astype(str)

    # ── 9. Date Dimension Columns ─────────────────────────────
    df["year"]       = df["date"].dt.year
    df["month"]      = df["date"].dt.month
    df["quarter"]    = df["date"].dt.quarter
    df["month_name"] = df["date"].dt.strftime("%b")
    df["year_month"] = df["date"].dt.strftime("%Y-%m")
    df["week"]       = df["date"].dt.isocalendar().week.astype(int)
    df["day_name"]   = df["date"].dt.day_name()

    return df

# ── Test on AAPL first ────────────────────────────────────────
aapl_raw      = master_df[master_df["ticker"] == "AAPL"].copy()
aapl_featured = engineer_features(aapl_raw)

original_cols = list(master_df.columns)
new_cols      = [c for c in aapl_featured.columns if c not in original_cols]

print(f"Original columns : {len(original_cols)}")
print(f"Features added   : {len(new_cols)}")
print(f"New features: {new_cols}")
print(f"\nSample output:")
print(aapl_featured[[
    "date", "close", "daily_return", "ma_50", "rsi_14",
    "volatility_30d", "drawdown", "golden_cross"
]].tail(5).to_string(index=False))

Original columns : 9
Features added   : 25
New features: ['daily_return', 'cumulative_return', 'ma_20', 'ma_50', 'ma_200', 'volatility_30d', 'volatility_ann', 'rsi_14', 'vwap', 'bb_upper', 'bb_lower', 'bb_width', 'bb_pct', 'drawdown', 'above_ma50', 'above_ma200', 'golden_cross', 'rsi_zone', 'year', 'month', 'quarter', 'month_name', 'year_month', 'week', 'day_name']

Sample output:
      date      close  daily_return  ma_50  rsi_14  volatility_30d  drawdown  golden_cross
2026-05-18 297.839996     -0.796061 266.78   82.55          1.5030   -0.7961             1
2026-05-19 298.970001      0.379400 267.57   84.06          1.4239   -0.4197             1
2026-05-20 302.250000      1.097100 268.40   84.81          1.3970    0.0000             1
2026-05-21 304.989990      0.906531 269.29   82.44          1.3986    0.0000             1
2026-05-22 308.820007      1.255785 270.36   91.10          1.4007    0.0000             1


In [8]:
# CELL 4 — Apply Feature Engineering to All 12 Tickers

logger.info("Starting feature engineering for all tickers...")

featured_list = []
failed        = []

for ticker in master_df['ticker'].unique():
    ticker_df = master_df[master_df['ticker'] == ticker].copy()

    try:
        featured = engineer_features(ticker_df)
        featured_list.append(featured)
        logger.info(
            f"[SUCCESS]{ticker}: {len(featured)} rows, "
            f"{len(featured.columns)} features"
        )

    except Exception as e:
        logger.error(f"[FAILED] {ticker} feature engineering failed: {e}")
        failed.append(ticker)

# Combine all
daily_featured = pd.concat(featured_list, ignore_index=True)
daily_featured = daily_featured.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

logger.info(f"\nFeature engineering complete")
logger.info(f"  Total rows    : {len(daily_featured):,}")
logger.info(f"  Total columns : {len(daily_featured.columns)}")
logger.info(f"  Failed        : {failed if failed else 'None'}")

print(f"\nFinal shape: {daily_featured.shape}")
print(f"Columns: {list(daily_featured.columns)}")

2026-05-24 22:34:20 | INFO | Starting feature engineering for all tickers...
2026-05-24 22:34:20 | INFO | [SUCCESS]AAPL: 1606 rows, 34 features
2026-05-24 22:34:20 | INFO | [SUCCESS]AMZN: 1606 rows, 34 features
2026-05-24 22:34:20 | INFO | [SUCCESS]BAC: 1606 rows, 34 features
2026-05-24 22:34:20 | INFO | [SUCCESS]CVX: 1606 rows, 34 features
2026-05-24 22:34:20 | INFO | [SUCCESS]GOOGL: 1606 rows, 34 features
2026-05-24 22:34:20 | INFO | [SUCCESS]GS: 1606 rows, 34 features
2026-05-24 22:34:21 | INFO | [SUCCESS]JNJ: 1606 rows, 34 features
2026-05-24 22:34:21 | INFO | [SUCCESS]JPM: 1606 rows, 34 features
2026-05-24 22:34:21 | INFO | [SUCCESS]MSFT: 1606 rows, 34 features
2026-05-24 22:34:21 | INFO | [SUCCESS]PFE: 1606 rows, 34 features
2026-05-24 22:34:21 | INFO | [SUCCESS]WMT: 1606 rows, 34 features
2026-05-24 22:34:21 | INFO | [SUCCESS]XOM: 1606 rows, 34 features
2026-05-24 22:34:21 | INFO | 
Feature engineering complete
2026-05-24 22:34:21 | INFO |   Total rows    : 19,272
2026-05-24 22:

In [10]:
# CELL 5 — Monthly Summary per Ticker

def create_monthly_summary(df):
    """
    Aggregates daily data to monthly level per ticker.
    Returns one row per (ticker, year, month) combination.
    """
    monthly = df.groupby(
        ["ticker", "sector", "year", "month", "quarter",
         "month_name", "year_month"]
    ).agg(
        open_price = ('open','first'),
        close_price = ("close", "last"),
        high_price  = ("high",  "max"),
        low_price   = ("low",   "min"),
        avg_close   = ("close", "mean"),

        # Volume
        total_volume = ("volume", "sum"),
        avg_volume   = ("volume", "mean"),

        # Return
        monthly_return   = ("daily_return", "sum"),
        avg_daily_return = ("daily_return", "mean"),

        # Risk
        monthly_volatility = ("volatility_30d", "last"),
        avg_volatility     = ("volatility_30d", "mean"),
        max_drawdown       = ("drawdown",        "min"),

        # Momentum indicators at month end
        end_rsi      = ("rsi_14",  "last"),
        avg_rsi      = ("rsi_14",  "mean"),
        end_ma50     = ("ma_50",   "last"),
        end_ma200    = ("ma_200",  "last"),
        golden_cross = ("golden_cross", "last"),

        # Count
        trading_days = ("date", "count")
    ).reset_index()

    # Month-over-month price change percentage
    monthly = monthly.sort_values(['ticker','year','month'])
    monthly['mom_pct_change'] = (
        monthly.groupby('ticker')['close_price'].pct_change() * 100
    ).round(4)

    # Round all numeric columns
    num_cols = monthly.select_dtypes(include=[np.number]).columns
    monthly[num_cols] = monthly[num_cols].round(4)

    return monthly

monthly_df = create_monthly_summary(daily_featured)

print(f"Monthly summary shape: {monthly_df.shape}")
print(f"Months covered       : {monthly_df['year_month'].nunique()}")
print(f"\nSample:")
print(monthly_df[[
    "ticker", "sector", "year_month",
    "close_price", "monthly_return", "monthly_volatility", "avg_rsi"
]].head(8).to_string(index=False))

Monthly summary shape: (924, 26)
Months covered       : 77

Sample:
ticker     sector year_month  close_price  monthly_return  monthly_volatility  avg_rsi
  AAPL Technology    2020-01      74.5399          3.3059              1.7679  62.3305
  AAPL Technology    2020-02      65.9901        -11.6068              2.2904  46.9989
  AAPL Technology    2020-03      61.3865         -2.7753              5.7696  39.2959
  AAPL Technology    2020-04      70.9243         15.4114              3.6438  62.0043
  AAPL Technology    2020-05      76.9596          8.3250              1.6423  69.6055
  AAPL Technology    2020-06      88.3024         14.2104              1.7647  70.0373
  AAPL Technology    2020-07     102.8839         16.0901              2.5395  65.7155
  AAPL Technology    2020-08     125.1654         20.0882              2.7373  76.0419


In [11]:
# CELL 6 — Sector-Level Monthly Summary

sector_monthly = daily_featured.groupby(
    ["sector", "year", "month", "quarter", "year_month"]
).agg(
    avg_close        = ("close",         "mean"),
    avg_daily_return = ("daily_return",  "mean"),
    avg_volatility   = ("volatility_30d","mean"),
    avg_rsi          = ("rsi_14",        "mean"),
    max_drawdown     = ("drawdown",      "min"),
    total_volume     = ("volume",        "sum"),
    ticker_count     = ("ticker",        "nunique"),
    avg_ma50         = ("ma_50",         "mean"),
    avg_ma200        = ("ma_200",        "mean"),
).reset_index()

# Round
num_cols = sector_monthly.select_dtypes(include=[np.number]).columns
sector_monthly[num_cols] = sector_monthly[num_cols].round(4)

print(f"Sector monthly shape: {sector_monthly.shape}")
print(sector_monthly.head(8).to_string(index=False))

Sector monthly shape: (385, 14)
  sector  year  month  quarter year_month  avg_close  avg_daily_return  avg_volatility  avg_rsi  max_drawdown  total_volume  ticker_count  avg_ma50  avg_ma200
Consumer  2020      1        1    2020-01    64.8346            0.0568          0.8698  35.9945       -4.1178    2077688500             2   65.0455    65.0455
Consumer  2020      2        1    2020-02    69.2954           -0.3168          1.3614  59.9145      -13.2000    2247444300             2   66.3337    66.3337
Consumer  2020      3        1    2020-03    64.2941            0.3115          3.2235  44.3052      -22.7447    4233046400             2   66.4482    66.3839
Consumer  2020      4        2    2020-04    74.9922            0.7598          3.8156  69.6040      -12.1476    3113675400             2   67.6010    66.9095
Consumer  2020      5        2    2020-05    78.9334            0.0431          2.2773  49.9532       -7.5974    2212057100             2   71.7383    69.3735
Consumer  2020

In [12]:
# CELL 7 — Save All Processed Datasets

# 1. Daily featured dataset — main analytical dataset
daily_path = f"../{PROCESSED_DATA_PATH}daily_features.csv"
daily_featured.to_csv(daily_path, index=False)
logger.info(f"Saved daily_features.csv: {len(daily_featured):,} rows")

# 2. Monthly ticker summary — for trend analysis and Power BI
monthly_path = f"../{PROCESSED_DATA_PATH}monthly_summary.csv"
monthly_df.to_csv(monthly_path, index=False)
logger.info(f"Saved monthly_summary.csv: {len(monthly_df):,} rows")

# 3. Sector monthly summary — for sector comparison
sector_path = f"../{PROCESSED_DATA_PATH}sector_monthly.csv"
sector_monthly.to_csv(sector_path, index=False)
logger.info(f"Saved sector_monthly.csv: {len(sector_monthly):,} rows")

print("\n✓ All processed datasets saved")
print(f"  daily_features.csv  : {len(daily_featured):,} rows")
print(f"  monthly_summary.csv : {len(monthly_df):,} rows")
print(f"  sector_monthly.csv  : {len(sector_monthly):,} rows")
print("\nNotebook 02 complete. Proceed to 03_eda_analysis.ipynb")

2026-05-24 22:44:15 | INFO | Saved daily_features.csv: 19,272 rows
2026-05-24 22:44:15 | INFO | Saved monthly_summary.csv: 924 rows
2026-05-24 22:44:15 | INFO | Saved sector_monthly.csv: 385 rows

✓ All processed datasets saved
  daily_features.csv  : 19,272 rows
  monthly_summary.csv : 924 rows
  sector_monthly.csv  : 385 rows

Notebook 02 complete. Proceed to 03_eda_analysis.ipynb
